# 🐼 Intermediate Pandas Masterclass

This notebook focuses on mastering Pandas operations essential for data scientists, including:

- `groupby()` and aggregations
- `merge()` strategies
- `pivot()` and `melt()`
- `apply()` with custom functions
- performance tuning with `categorical`, `vectorized ops`, and `query()`

Custom datasets are created to simulate real-world data wrangling tasks.

In [1]:

import pandas as pd
import numpy as np

# Setup random seed for reproducibility
np.random.seed(42)


## 🏗️ Step 1: Create Custom Dataset

In [4]:

# Create synthetic ecommerce sales dataset
customers = ['Alice', 'Bob', 'Charlie', 'David', 'Eva']
products = ['Laptop', 'Smartphone', 'Tablet', 'Monitor', 'Headphones']

data = {
    'Customer': np.random.choice(customers, size=20),
    'Product': np.random.choice(products, size=20),
    'Units': np.random.randint(1, 5, size=20),
    'Unit_Price': np.random.randint(100, 1000, size=20),
    'Region': np.random.choice(['North', 'South', 'East', 'West'], size=20),
    'Date': pd.date_range(start='2024-01-01', periods=20, freq='D')
}

df = pd.DataFrame(data)
df['Total'] = df['Units'] * df['Unit_Price']
df.head()


,Customer,Product,Units,Unit_Price,Region,Date,Total
0,David,Smartphone,4,305,West,2024-01-01,1220
1,Eva,Headphones,1,180,East,2024-01-02,180
2,Charlie,Monitor,4,661,West,2024-01-03,2644
3,Eva,Laptop,2,971,South,2024-01-04,1942
4,Eva,Laptop,2,487,East,2024-01-05,974


## 📊 Step 2: GroupBy and Aggregations

In [6]:

# Total units sold by customer
df.groupby('Customer')['Units'].sum().reset_index()

# Multiple aggregations
df.groupby('Product').agg({
    'Units': 'sum',
    'Total': ['mean', 'sum']
})


Units        Total      
             sum         mean   sum
Product                            
Headphones     9  1334.666667  4004
Laptop        11  1506.750000  6027
Monitor       14  1860.200000  9301
Smartphone     9  1812.000000  5436
Tablet        11  1424.600000  7123

## 🔗 Step 3: Merge

In [8]:

# Add customer info (merge)
customer_info = pd.DataFrame({
    'Customer': customers,
    'Loyalty_Level': ['Gold', 'Silver', 'Platinum', 'Gold', 'Silver']
})

merged = pd.merge(df, customer_info, on='Customer', how='left')
merged.head()


,Customer,Product,Units,Unit_Price,Region,Date,Total,Loyalty_Level
0,David,Smartphone,4,305,West,2024-01-01,1220,Gold
1,Eva,Headphones,1,180,East,2024-01-02,180,Silver
2,Charlie,Monitor,4,661,West,2024-01-03,2644,Platinum
3,Eva,Laptop,2,971,South,2024-01-04,1942,Silver
4,Eva,Laptop,2,487,East,2024-01-05,974,Silver


## 🔄 Step 4: Pivot and Melt

In [14]:

# Pivot: Total sales per product per region
pivot_table = df.pivot_table(values='Total', index='Region', columns='Product', aggfunc='sum', fill_value=0)
pivot_table

# Melt back into long format
df_melted = pivot_table.reset_index().melt(id_vars='Region', var_name='Product', value_name='Total_Sales')
df_melted.head()


,Region,Product,Total_Sales
0,East,Headphones,180
1,North,Headphones,1204
2,South,Headphones,0
3,West,Headphones,2620
4,East,Laptop,2978


## 🧠 Step 5: Apply Custom Functions

In [16]:

# Flag high-value transactions
def flag_high_value(row):
    return 'YES' if row['Total'] > 2000 else 'NO'

df['High_Value'] = df.apply(flag_high_value, axis=1)
df[['Customer', 'Product', 'Total', 'High_Value']].head()


,Customer,Product,Total,High_Value
0,David,Smartphone,1220,NO
1,Eva,Headphones,180,NO
2,Charlie,Monitor,2644,YES
3,Eva,Laptop,1942,NO
4,Eva,Laptop,974,NO


## ⚡ Step 6: Performance Tips

In [18]:

# Convert string column to category
df['Region'] = df['Region'].astype('category')

# Use query for filtering
df.query('Units >= 3 and Total > 1500')


,Customer,Product,Units,Unit_Price,Region,Date,Total,High_Value
2,Charlie,Monitor,4,661,West,2024-01-03,2644,YES
10,David,Tablet,4,921,West,2024-01-11,3684,YES
11,Charlie,Monitor,4,576,North,2024-01-12,2304,YES
12,Eva,Monitor,3,802,South,2024-01-13,2406,YES
13,Bob,Laptop,4,501,East,2024-01-14,2004,YES
14,David,Tablet,3,829,North,2024-01-15,2487,YES
15,Bob,Headphones,4,655,West,2024-01-16,2620,YES
19,David,Smartphone,3,962,West,2024-01-20,2886,YES


## 🎯 Interview Questions + Answers


**Q1: What is the difference between `pivot()` and `pivot_table()` in Pandas?**  
A: `pivot()` doesn't allow duplicate entries for the index/columns and throws an error, while `pivot_table()` can handle duplicates using an aggregation function.

**Q2: When would you use `apply()` over vectorized operations?**  
A: When the operation is too complex for native functions and requires row-wise or column-wise custom logic, though it's slower than vectorized methods.

**Q3: How does converting a column to `category` help performance?**  
A: Reduces memory usage and improves speed for operations like grouping and filtering, especially with repeated string values.

**Q4: Explain the types of joins in Pandas merge and when to use them.**  
A: `inner` (intersection), `left` (preserve left), `right` (preserve right), `outer` (union). Use based on how much info you want to retain from either table.
